# s26 — Mailbox / Teammates

**What this teaches:** how to give one agent a *persistent inbox* so it can send and receive messages addressed to other named teammates. The `teammates` package exposes three tools — `teammate_tell`, `teammate_ask`, `teammate_check` — that wrap a small FSM (idle / requesting / waiting / responding) and a pluggable backend (in-memory by default, Redis in [s29_redis](../s29_redis/s29_redis.ipynb)).

**Concept anchor:** a mailbox is a *named addressable queue* scoped to one session. The agent does not need to know whether the recipient is another goroutine, another process, or a future agent — the FSM and backend hide that. The same loop from [s01_loop](../s01_loop/s01_loop.ipynb) is unchanged; we are only adding three new tools to the leader.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars before launching Jupyter, e.g.:
  ```
  export YOKE_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export YOKE_MODEL=qwen2.5-coder
  ```
  Defaults to a local Ollama. Swap in `anthropic` / `openai` / `gemini` for hosted providers — see [docs/providers.md](../../docs/providers.md).
- No external service required: the default backend is in-memory and per-session.

## 1. Imports

We add `internal/teammates` to the imports from [s01_loop](../s01_loop/s01_loop.ipynb). That package owns both the FSM-backed `Agent` value and the mailbox backend abstraction.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/stream"
    "github.com/blouargant/yoke/internal/teammates"
)

## 2. Helper

Notebook-safe `must` — panics instead of `os.Exit` so the GoNB kernel survives an error.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Pick a backend and create the mailbox

`teammates.ChooseBackend()` resolves to the in-memory backend by default. Set `MAILBOX_BACKEND=redis` (and `REDIS_URL`) to get a distributed one — that variation is the whole point of [s29_redis](../s29_redis/s29_redis.ipynb). `NewAgent("lead", be)` creates a named teammate; its identity is what other agents will use as the `to` field when they send messages.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
be, err := teammates.ChooseBackend()
must(err)
defer be.Close()
me := teammates.NewAgent("lead", be)
fmt.Println("mailbox ready for:", "lead")

## 4. Wire the agent with mailbox tools

`me.Tools()` returns the three mailbox tools bound to *this* agent identity. The instruction tells the model exactly how to use them so we get a deterministic demo. Behind the scenes each tool transitions the FSM — see [s27_fsm](../s27_fsm/s27_fsm.ipynb) for a peek at the state diagram.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s26_mailbox",
    Description: "Lead agent with teammate mailbox.",
    Instruction: "Use teammate_tell with fields `to` and `body` to send a one-way message to 'reviewer', then teammate_check to read your own mailbox.",
    Model:       llm,
    Tools:       me.Tools(),
})
must(err)

## 5. Run the loop

The prompt explicitly asks the model to call `teammate_tell` (one-way send) then `teammate_check` (read own inbox). With no real reviewer process running, the inbox check should return empty — that is the expected, observable outcome of a one-way send when no one is on the other end.

In [ ]:
r, err := agentkit.Runner("s26", a)
must(err)
prompt := "Call teammate_tell with to='reviewer' and body='please review PR #42', then call teammate_check on your own mailbox."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- Two tool calls in the trace: `teammate_tell` followed by `teammate_check`. The leader's loop is identical to [s01_loop](../s01_loop/s01_loop.ipynb); only the tool set changed.
- Mailbox state is **per session** — restart the notebook and the queue is empty again. For durability across processes, switch to the Redis backend in [s29_redis](../s29_redis/s29_redis.ipynb).
- Compare with [s27_fsm](../s27_fsm/s27_fsm.ipynb), which exercises the full ask/reply round-trip and prints the FSM state transitions.

## Try it yourself

1. Spawn a second `teammates.NewAgent("reviewer", be)` in a separate cell and call `rev.Check(ctx, ...)` before re-running the loop — you should see the message arrive.
2. Replace `teammate_tell` with `teammate_ask` in the prompt and observe how the FSM transitions to `WAITING` (see [s27_fsm](../s27_fsm/s27_fsm.ipynb)).